In [5]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
        .appName ('TidarMart')     
        .master('local[2]')  
        .config('spark.driver.memory', '1536m')     
        .config('spark.sql.shuffle.partitions', '4')     
        .config('spark.sql.adaptive.enabled', 'true')    
        .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

SparkSession berhasil dibuat!
Versi Spark: 3.5.9


**Tahap 1 – Penyimpanan Terdistribusi dan Ingesti (HDFS)**

Pada tahap ini dilakukan pembacaan data dari HDFS menggunakan PySpark. Data pelanggan, produk, dan transaksi diperiksa untuk mengetahui jumlah data, struktur kolom, serta isi datanya. Pemeriksaan ini bertujuan untuk memahami kondisi awal data sebelum dilakukan proses pengolahan lebih lanjut.

Membuat Diretori HDFS 

In [6]:
!hdfs dfs -mkdir -p /tidarmart/raw/transaksi
!hdfs dfs -mkdir -p /tidarmart/raw/pelanggan
!hdfs dfs -mkdir -p /tidarmart/raw/produk
!hdfs dfs -mkdir -p /tidarmart/raw/strem_input
!hdfs dfs -mkdir -p /tidarmart/raw/curated
!hdfs dfs -mkdir -p /tidarmart/raw/models

In [7]:
!hdfs dfs -ls -R /tidarmart

drwxr-xr-x   - fahma supergroup          0 2026-11-26 08:50 /tidarmart/raw
drwxr-xr-x   - fahma supergroup          0 2026-11-26 07:59 /tidarmart/raw/curated
drwxr-xr-x   - fahma supergroup          0 2026-11-26 08:00 /tidarmart/raw/models
drwxr-xr-x   - fahma supergroup          0 2026-11-26 08:08 /tidarmart/raw/pelanggan
-rw-r--r--   1 fahma supergroup    6307840 2026-11-26 08:08 /tidarmart/raw/pelanggan/pelanggan.json
drwxr-xr-x   - fahma supergroup          0 2026-11-26 08:08 /tidarmart/raw/produk
-rw-r--r--   1 fahma supergroup     700615 2026-11-26 08:08 /tidarmart/raw/produk/produk.json
drwxr-xr-x   - fahma supergroup          0 2026-11-26 07:59 /tidarmart/raw/stream_input
drwxr-xr-x   - fahma supergroup          0 2026-11-26 08:50 /tidarmart/raw/strem_input
drwxr-xr-x   - fahma supergroup          0 2026-11-26 08:07 /tidarmart/raw/transaksi
-rw-r--r--   1 fahma supergroup   49068656 2026-11-26 08:07 /tidarmart/raw/transaksi/transaksi_part1.csv
-rw-r--r--   1 fahma supergroup   

Menguggah Data ke HDFS

In [20]:
!hdfs dfs -put -f /home/fahma/praktikum-bigdata/proyek-akhir/data_proyek/transaksi/*.csv /tidarmart/raw/transaksi/
!hdfs dfs -put -f /home/fahma/praktikum-bigdata/proyek-akhir/data_proyek/pelanggan.json /tidarmart/raw/pelanggan/
!hdfs dfs -put -f /home/fahma/praktikum-bigdata/proyek-akhir/data_proyek/produk.json /tidarmart/raw/produk/
!hdfs dfs -put -f /home/fahma/praktikum-bigdata/proyek-akhir/data_proyek/stream_input/*.csv /tidarmart/raw/stream_input/

Memeriksa Lokasi Direktori yang Sedang Digunakan

In [88]:
pwd

'/home/fahma/praktikum-bigdata/proyek-akhir'

Memeriksa Data di HDFS Beserta Seluruh Subdirektori

In [21]:
!hdfs dfs -ls -R /tidarmart

drwxr-xr-x   - fahma supergroup          0 2026-11-26 08:50 /tidarmart/raw
drwxr-xr-x   - fahma supergroup          0 2026-11-26 07:59 /tidarmart/raw/curated
drwxr-xr-x   - fahma supergroup          0 2026-11-26 08:00 /tidarmart/raw/models
drwxr-xr-x   - fahma supergroup          0 2026-11-26 09:10 /tidarmart/raw/pelanggan
-rw-r--r--   1 fahma supergroup    6307840 2026-11-26 09:10 /tidarmart/raw/pelanggan/pelanggan.json
drwxr-xr-x   - fahma supergroup          0 2026-11-26 09:10 /tidarmart/raw/produk
-rw-r--r--   1 fahma supergroup     700615 2026-11-26 09:10 /tidarmart/raw/produk/produk.json
drwxr-xr-x   - fahma supergroup          0 2026-11-26 09:10 /tidarmart/raw/stream_input
-rw-r--r--   1 fahma supergroup     490170 2026-11-26 09:10 /tidarmart/raw/stream_input/stream_01.csv
-rw-r--r--   1 fahma supergroup     490310 2026-11-26 09:10 /tidarmart/raw/stream_input/stream_02.csv
-rw-r--r--   1 fahma supergroup     490723 2026-11-26 09:10 /tidarmart/raw/stream_input/stream_03.csv
-rw-r

Memeriksa Isi setiap Direktori HDFS

In [22]:
!hdfs dfs -ls /tidarmart/raw/transaksi
!hdfs dfs -ls /tidarmart/raw/pelanggan
!hdfs dfs -ls /tidarmart/raw/produk
!hdfs dfs -ls /tidarmart/raw/stream_input

Found 4 items
-rw-r--r--   1 fahma supergroup   49068656 2026-11-26 09:10 /tidarmart/raw/transaksi/transaksi_part1.csv
-rw-r--r--   1 fahma supergroup   49063997 2026-11-26 09:10 /tidarmart/raw/transaksi/transaksi_part2.csv
-rw-r--r--   1 fahma supergroup   49062882 2026-11-26 09:10 /tidarmart/raw/transaksi/transaksi_part3.csv
-rw-r--r--   1 fahma supergroup   49070313 2026-11-26 09:10 /tidarmart/raw/transaksi/transaksi_part4.csv
Found 1 items
-rw-r--r--   1 fahma supergroup    6307840 2026-11-26 09:10 /tidarmart/raw/pelanggan/pelanggan.json
Found 1 items
-rw-r--r--   1 fahma supergroup     700615 2026-11-26 09:10 /tidarmart/raw/produk/produk.json
Found 20 items
-rw-r--r--   1 fahma supergroup     490170 2026-11-26 09:10 /tidarmart/raw/stream_input/stream_01.csv
-rw-r--r--   1 fahma supergroup     490310 2026-11-26 09:10 /tidarmart/raw/stream_input/stream_02.csv
-rw-r--r--   1 fahma supergroup     490723 2026-11-26 09:10 /tidarmart/raw/stream_input/stream_03.csv
-rw-r--r--   1 fahma su

Memeriksa Penggunaan Ruang Penyimpanan HDFS

In [89]:
!hdfs dfs -du -h /tidarmart

203.2 M  203.2 M  /tidarmart/raw


Memeriksa Kondisi dan Blok Data HDFS

In [91]:
!hdfs fsck /tidarmart -files -blocks -locations

Connecting to namenode via http://localhost:9870/fsck?ugi=fahma&files=1&blocks=1&locations=1&path=%2Ftidarmart
FSCK started by fahma (auth:SIMPLE) from /127.0.0.1 for path /tidarmart at Thu Nov 26 19:04:53 WIB 2026

/tidarmart <dir>
/tidarmart/raw <dir>
/tidarmart/raw/curated <dir>
/tidarmart/raw/models <dir>
/tidarmart/raw/pelanggan <dir>
/tidarmart/raw/pelanggan/pelanggan.json 6307840 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-111730338-127.0.1.1-1795610580917:blk_1073741850_1026 len=6307840 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-6e060d9d-c7c0-453f-858e-f2e75af20d5d,DISK]]

/tidarmart/raw/produk <dir>
/tidarmart/raw/produk/produk.json 700615 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-111730338-127.0.1.1-1795610580917:blk_1073741851_1027 len=700615 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-6e060d9d-c7c0-453f-858e-f2e75af20d5d,DISK]]

/tidarmart/raw/stream_input <dir>
/tidarmart/raw/stream_input/stream_01.csv 490170 bytes, replic

Membaca Seluruh data HDFS

In [38]:
HDFS = "hdfs://localhost:9000/tidarmart"
df_trx = spark.read.csv(f"{HDFS}/raw/transaksi/*.csv", header=True)
print("Jumlah baris:", df_trx.count())
df_trx.printSchema()
df_trx.show(5)

Jumlah baris: 2010000
root
 |-- order_id: string (nullable = true)
 |-- tanggal: string (nullable = true)
 |-- pelanggan_id: string (nullable = true)
 |-- produk_id: string (nullable = true)
 |-- qty: string (nullable = true)
 |-- harga_satuan: string (nullable = true)
 |-- diskon_persen: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- berat_kg: string (nullable = true)
 |-- jarak_km: string (nullable = true)
 |-- lama_kirim_hari: string (nullable = true)
 |-- status_pesanan: string (nullable = true)

+----------+-------------------+------------+---------+---+------------+-------------+-----------------+----------+--------+--------+---------------+--------------+
|  order_id|            tanggal|pelanggan_id|produk_id|qty|harga_satuan|diskon_persen|metode_pembayaran|      kota|berat_kg|jarak_km|lama_kirim_hari|status_pesanan|
+----------+-------------------+------------+---------+---+------------+-------------+----------

In [43]:
from pyspark.sql.types import StructType, StructField, StringType
skema_pel = StructType([
    StructField("pelanggan_id", StringType()),
    StructField("nama", StringType()),
    StructField("kota", StringType()),
    StructField("tanggal_daftar", StringType()),
    StructField("membership", StringType()),
])

In [49]:
df_pel = spark.read.schema(skema_pel).json(f"{HDFS}/raw/pelanggan/pelanggan.json")
print("Jumlah pelanggan:", df_pel.count())
df_pel.printSchema()
df_pel.show(7, truncate=False)

Jumlah pelanggan: 50000
root
 |-- pelanggan_id: string (nullable = true)
 |-- nama: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- tanggal_daftar: string (nullable = true)
 |-- membership: string (nullable = true)

+------------+-----------+----------+--------------+----------+
|pelanggan_id|nama       |kota      |tanggal_daftar|membership|
+------------+-----------+----------+--------------+----------+
|1           |Pelanggan-1|Semarang  |2026-04-06    |Gold      |
|2           |Pelanggan-2|Temanggung|2025-06-22    |Silver    |
|3           |Pelanggan-3|Solo      |2026-05-28    |Silver    |
|4           |Pelanggan-4|Wonosobo  |2026-02-23    |Silver    |
|5           |Pelanggan-5|Magelang  |2026-01-19    |Silver    |
|6           |Pelanggan-6|Semarang  |2024-07-17    |Silver    |
|7           |Pelanggan-7|Semarang  |2024-03-16    |Silver    |
+------------+-----------+----------+--------------+----------+
only showing top 7 rows



In [54]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, ArrayType
skema_prod = StructType([
    StructField("produk_id", StringType()),
    StructField("nama_produk", StringType()),
    StructField("kategori", StringType()),
    StructField("harga_dasar", DoubleType()),
    StructField("garansi_bulan", LongType()),
    StructField("voltas", LongType()),
    StructField("ukuran", StringType()),
    StrkuctField("bahan", StringType()),
    StructField("kadaluarsa_hari", LongType()),
    StructField("penerbit", StringType()),
    StructField("tags", ArrayType(StringType())),
])

In [55]:
df_prod = spark.read.schema(skema_prod).json(f"{HDFS}/raw/produk/produk.json")
print("Jumlah produk:", df_prod.count())
df_prod.printSchema()
df_prod.show(7, truncate=False)

Jumlah produk: 5000
root
 |-- produk_id: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga_dasar: double (nullable = true)
 |-- garansi_bulan: long (nullable = true)
 |-- voltas: long (nullable = true)
 |-- ukuran: string (nullable = true)
 |-- bahan: string (nullable = true)
 |-- kadaluarsa_hari: long (nullable = true)
 |-- penerbit: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)

+---------+--------------+------------+-----------+-------------+------+------+---------+---------------+--------+------------------------+
|produk_id|nama_produk   |kategori    |harga_dasar|garansi_bulan|voltas|ukuran|bahan    |kadaluarsa_hari|penerbit|tags                    |
+---------+--------------+------------+-----------+-------------+------+------+---------+---------------+--------+------------------------+
|1        |Fashion-1     |Fashion     |350000.0   |NULL         |

**Pemisahan direktori raw dan curated**

Direktori raw digunkan untuk menyimpan dat asli sebelum proses pembersihan. SEdangkan direktori curted digunakan untuk menyimpan data yang telah melalui proses ETL dan pembersihan. Pemisahan ini bertujuan agar data  asli tetap terjaga dan proses pengolahan lebih terstuktur.

**Arti replikasi = 1**

Replikasi 1 berarti setiap blok hanya memiliki satu salinan. Jika terjadi kerusakan pada penyimpanan, tidak ada salinan lain yang bisa digunakan untuk menggantikan data tersebut. Sedangkan pada sistem dengan beberapa DataNode, data dapat disimpan dalam beberapa salinan agar lebih aman.

**Tahap 2 – Profiling Kualitas Data (Veracity)**

Pada tahap ini dilakukan pemeriksaan kualitas data untuk mengetahui kondisi data sebelum dibersihkan. Pemeriksaan meliputi jumlah baris, data unik, nilai kosong (null), nilai negatif, serta data yang memiliki nilai tidak wajar. 

Menghitung Jumlah Baris

In [57]:
total_baris = df_trx.count()
print("total_baris :", total_baris)

total_baris : 2010000


Menghitung Jumlah order_id unik

In [129]:
order_unik = df_trx.select("order_id").distinct().count()
total_baris = df_trx.count()
dup = total_baris - order_unik

print("order_id unik", order_unik)
print("jumlah order ID berulang", dup)

order_id unik 2000000
jumlah order ID berulang 10000


Menghitung Presentase Data Kosong

In [62]:
from pyspark.sql.functions import col

total = df_trx.count()

for c in df_trx.columns:
    kosong = df_trx.filter(col(c).isNull()).count()
    persen = (kosong / total) * 100
    print(c,":", kosong, "data kosong (", round(persen, 2), "%)")

order_id : 0 data kosong ( 0.0 %)


tanggal : 0 data kosong ( 0.0 %)


pelanggan_id : 0 data kosong ( 0.0 %)


produk_id : 0 data kosong ( 0.0 %)


qty : 6131 data kosong ( 0.31 %)


harga_satuan : 6003 data kosong ( 0.3 %)


... (2 outputs left out)

kota : 0 data kosong ( 0.0 %)


berat_kg : 0 data kosong ( 0.0 %)


jarak_km : 0 data kosong ( 0.0 %)


lama_kirim_hari : 0 data kosong ( 0.0 %)
status_pesanan : 0 data kosong ( 0.0 %)


Menhitung Jumlah Harga Negatif

In [66]:
harga_negatif = df_trx.filter(col("harga_satuan") < 0).count()
print("jumlah harga negatif:", harga_negatif)

[Stage 100:>                                                        (0 + 2) / 2]

jumlah harga negatif: 1968


Distribusi Qty

In [77]:
print("Distribusi Qty")
df_trx.groupBy("qty").count().orderBy("qty").show()

Distribusi Qty


[Stage 118:>                                                        (0 + 2) / 2]

+-----+------+
|  qty| count|
+-----+------+
| NULL|  6131|
|  1.0|286837|
|  2.0|286145|
|  3.0|287568|
|  4.0|285605|
|  5.0|285267|
|  6.0|285425|
|  7.0|285999|
|999.0|  1023|
+-----+------+



In [72]:
df_qty = df_trx.select(col("qty").cast("double").alias("qty"))
kuartil_qty = df_qty.approxQuantile("qty", [0.25, 0.5, 0.75, 0.99, 0.999, 1.0], 0.001)

Q1 = kuartil_qty[0]
Q2 = kuartil_qty[1]
Q3 = kuartil_qty[2]

print("Q1:", Q1)
print("Q2 (median):", Q2)
print("Q3:", Q3)

[Stage 108:>                                                        (0 + 2) / 2]

Q1: 2.0
Q2 (median): 4.0
Q3: 6.0


In [73]:
IQR = Q3 - Q1
batas_bawah = Q1 - (1.5 * IQR)
batas_atas = Q3 + (1.5 * IQR)

print("IQR:", IQR)
print("Batas bawah:", batas_bawah)
print("Batas atas:", batas_atas)

IQR: 4.0
Batas bawah: -4.0
Batas atas: 12.0


In [75]:
from pyspark.sql.functions import col
jumlah_outlier_iqr = df_qty.filter(
    (col("qty") < batas_bawah) |
    (col("qty") > batas_atas)).count()

print("jumlah outlier IQR:", jumlah_outlier_iqr)

jumlah_qty_999 = df_trx.filter(col("qty") == 999).count()
print("jumlah_qty_999:", jumlah_qty_999)

jumlah outlier IQR: 1023
jumlah_qty_999: 1023


Variasi Penulisan Kota

In [79]:
df_trx.groupBy("kota") \
  .count() \
  .orderBy("kota") \
  .show(100, truncate=False)

[Stage 121:>                                                        (0 + 2) / 2]

+----------+------+
|kota      |count |
+----------+------+
|KEBUMEN   |6280  |
|Kebumen   |238457|
|MAGELANG  |6379  |
|Magelang  |238449|
|PURWOREJO |6311  |
|Purworejo |238337|
|SEMARANG  |6254  |
|SOLO      |6335  |
|Semarang  |238446|
|Solo      |238603|
|TEMANGGUNG|6199  |
|Temanggung|238968|
|WONOSOBO  |6435  |
|Wonosobo  |239362|
|YOGYAKARTA|6340  |
|Yogyakarta|238364|
|kebumen   |6306  |
|magelang  |6240  |
|purworejo |6395  |
|semarang  |6374  |
|solo      |6367  |
|temanggung|6233  |
|wonosobo  |6277  |
|yogyakarta|6289  |
+----------+------+



In [137]:
kota_raw = df_trx.select("kota").distinct().count()
print("Jumlah variasi penulisan kota:", kota_raw)

[Stage 262:============================>                            (1 + 1) / 2]

Jumlah variasi penulisan kota: 24


In [82]:
hasil_kategori = df_prod.groupBy("kategori").count().orderBy("kategori")
hasil_kategori.show()

+------------+-----+
|    kategori|count|
+------------+-----+
|        Buku|  618|
|  Elektronik|  646|
|     Fashion|  615|
|   Kesehatan|  642|
|     Makanan|  604|
|    Olahraga|  631|
|    Otomotif|  594|
|Rumah Tangga|  650|
+------------+-----+



Tabel Ringkasan

In [138]:
ringkasan = [
    ("Total baris", total_baris),
    ("order_id unik", order_unik),
    ("Baris duplikat", dup),
    ("qty kosong", kosong["qty"]),
    ("harga_satuan kosong", kosong["harga_satuan"]),
    ("Harga negatif", harga_negatif),
    ("Qty outlier", jumlah_outlier_iqr),
    ("Variasi penulisan kota", kota_raw)]

df_ringkasan = spark.createDataFrame(ringkasan, ["metrik", "nilai"])
df_ringkasan.show(truncate=False)

+----------------------+-------+
|metrik                |nilai  |
+----------------------+-------+
|Total baris           |2010000|
|order_id unik         |2000000|
|Baris duplikat        |10000  |
|qty kosong            |6131   |
|harga_satuan kosong   |6003   |
|Harga negatif         |1968   |
|Qty outlier           |1023   |
|Variasi penulisan kota|24     |
+----------------------+-------+



Profiling 

In [99]:
from pyspark.sql.functions import col

def hitung_presentase(data, atribut):
    total = data.count()
    ada_atribut  = data.filter(col(atribut).isNotNull()).count()
    hasil = (ada_atribut / total) * 100
    return round(hasil, 2)

In [106]:
df_buku = df_prod.filter(col("kategori") == "Buku")
print("Presentase penerbit:", hitung_presentase(df_buku, "penerbit"), "%")  

Presentase penerbit: 100.0 %


In [107]:
df_elektronik = df_prod.filter(col("kategori") == "Elektronik")
print("Presentase garansi:", hitung_presentase(df_elektronik, "garansi_bulan"), "%")  
print("Presentase voltas:", hitung_presentase(df_elektronik, "voltas"), "%")

Presentase garansi: 100.0 %
Presentase voltas: 0.0 %


In [149]:
f_elektronik = df_prod.filter(col("kategori") == "Elektronik").select("produk_id", "garansi_bulan", "voltas").show(10, truncate=False)

+---------+-------------+------+
|produk_id|garansi_bulan|voltas|
+---------+-------------+------+
|4        |6            |NULL  |
|18       |24           |NULL  |
|32       |12           |NULL  |
|39       |6            |NULL  |
|42       |12           |NULL  |
|47       |6            |NULL  |
|50       |6            |NULL  |
|52       |24           |NULL  |
|59       |24           |NULL  |
|66       |24           |NULL  |
+---------+-------------+------+
only showing top 10 rows



In [150]:
df_elektronik.filter(col("voltas").isNotNull()).count()

0

In [108]:
df_fashion = df_prod.filter(col("kategori") == "Fashion")
print("Presentase ukuran:", hitung_presentase(df_fashion, "ukuran"), "%")  
print("Presentase bahan:", hitung_presentase(df_fashion, "bahan"), "%")

Presentase ukuran: 100.0 %
Presentase bahan: 100.0 %


In [109]:
df_kesehatan = df_prod.filter(col("kategori") == "Kesehatan")
print("Jumlah produk kesehatan:", df_kesehatan.count())

Jumlah produk kesehatan: 642


In [110]:
df_makanan = df_prod.filter(col("kategori") == "Makanan")
print("Presentase kadaluarsa:", hitung_presentase(df_makanan, "kadaluarsa_hari"), "%")  

Presentase kadaluarsa: 100.0 %


In [111]:
df_olahraga = df_prod.filter(col("kategori") == "Olahraga")
print("Jumlah produk olahraga:", df_olahraga.count())

Jumlah produk olahraga: 631


In [112]:
df_otomotif = df_prod.filter(col("kategori") == "Otomotif")
print("Jumlah produk otomotif:", df_otomotif.count())

Jumlah produk otomotif: 594


In [113]:
df_rumah_tangga = df_prod.filter(col("kategori") == "Rumah Tangga")
print("Jumlah produk rumah tangga:", df_rumah_tangga.count())

Jumlah produk rumah tangga: 650


In [132]:
import pyspark.sql.functions as F
kosong = {}
for c in df_trx.columns:
    kosong[c] = df_trx.filter(F.col(c).isNull()).count()

In [139]:
jumlah_tags = df_prod.where("tags IS NOT NULL").count()
tanpa_tags = df_prod.where("tags IS NULL").count()

print("Produk dengan tags:", jumlah_tags)
print("Produk tanpa tags:", tanpa_tags)

Produk dengan tags: 1531
Produk tanpa tags: 3469


In [143]:
from pyspark.sql.functions import col

print("jumlah data pelanggan:", df_pel.count())
print("jumlah ID pelanggan unik:", df_pel.select("pelanggan_id").distinct().count())
print("distribusi membership:")
df_pel.groupBy("membership").count().show()

jumlah data pelanggan: 50000
jumlah ID pelanggan unik: 50000
distribusi membership:
+----------+-----+
|membership|count|
+----------+-----+
|      Gold|15208|
|    Silver|29885|
|  Platinum| 4907|
+----------+-----+



In [147]:
for c in df_pel.columns:
    kosong = df_pel.filter(col(c).isNull()).count()
    persen = (kosong / df_pel.count())*100 

print(c, ":", kosong, "(", round(persen, 2), "%)")

membership : 0 ( 0.0 %)


**Tahap 3 – ETL dan Penyimpanan Kolumnar (Parquet)**

Pada tahap in, data transaksi dibersihkan berdasarkan hasil profiling pada Tahap 2.Selatelah dibersihkan, akan ditambahkan beberapa kolom baru untuk mendukung analisis. Selanjutnya, data disimpan ke HDFS dalam format Parquet agar lebih efisien untuk proses pengelolahan dan analisis data.

Membersihkan transaksi sesuai tahab 2

In [180]:
# Deduplikasi
df_trx_bersih = df_trx.dropDuplicates()
print("jumlah baris sebelum:", df_trx.count())
print("jumlah baris sesudah:", df_trx_bersih.count())

jumlah baris sebelum: 2010000


[Stage 454:============================>                            (2 + 2) / 4]

jumlah baris sesudah: 2000000


**Alasan** : Deduplikasi dilakukan untuk menghapus baris transaksi yang memiliki seluruh nilai kolom sama. HAl inin bertujuan menghindari perhitungan transaksi ganda yang dapat memengaruhi hasil analisis

In [158]:
#identifikasi nilai kosong
df_trx_bersih.filter("qty IS NULL OR harga_satuan IS NULL").show(5, truncate=False)

[Stage 364:>                                                        (0 + 2) / 2]

+----------+-------------------+------------+---------+----+------------+-------------+-----------------+----------+--------+--------+---------------+--------------+
|order_id  |tanggal            |pelanggan_id|produk_id|qty |harga_satuan|diskon_persen|metode_pembayaran|kota      |berat_kg|jarak_km|lama_kirim_hari|status_pesanan|
+----------+-------------------+------------+---------+----+------------+-------------+-----------------+----------+--------+--------+---------------+--------------+
|ORD4001165|2026-06-20 20:28:22|17617       |276      |NULL|60000.0     |15           |Transfer Bank    |Solo      |11.92   |108.4   |4              |Selesai       |
|ORD4001560|2026-01-30 06:16:04|38247       |1410     |1.0 |NULL        |15           |Transfer Bank    |Solo      |11.83   |112.7   |4              |Selesai       |
|ORD4001982|2026-06-01 01:26:17|7797        |4052     |7.0 |NULL        |15           |Transfer Bank    |Temanggung|19.6    |41.0    |2              |Selesai       |
|ORD

In [159]:
print("qty kosong:", df_trx_bersih.filter("qty IS NULL").count())
print("harga_satuan kosong:", df_trx_bersih.filter("harga_satuan IS NULL").count())

qty kosong: 6096


[Stage 373:>                                                        (0 + 2) / 2]

harga_satuan kosong: 5975


In [181]:
# Menangani nilai kosong pada qty
df_trx_bersih = df_trx_bersih.filter(col("qty").isNotNull())
print("Baris setelah penanganan qty kosong:", df_trx_bersih.count())

[Stage 460:============================>                            (2 + 2) / 4]

Baris setelah penanganan qty kosong: 1993904


**Alasan** : Baris dengan nilai qty kosong dihapus karena jumlah barang tidak diketahui. Nilai tersebut diperlukan untuk menghitug total nilai transaksi sehingga tidak dapat dibiarkan kosong.

In [182]:
# menangani harga_satuan kosong
df_trx_bersih = df_trx_bersih.join(df_prod.select("produk_id","harga_dasar"), on="produk_id", how="left")

In [183]:
from pyspark.sql.functions import when

df_trx_bersih = df_trx_bersih.withColumn("harga_satuan", when(col("harga_satuan").isNull(), col("harga_dasar")).otherwise(col("harga_satuan")))
df_trx_bersih = df_trx_bersih.drop("harga_dasar")

print("jumlah baris setelah penanganan nilai kosong:", df_trx_bersih.count())
print("Harga satuan kosong:", df_trx_bersih.filter("harga_satuan IS NULL").count())

jumlah baris setelah penanganan nilai kosong: 1993904


[Stage 474:============================>                            (2 + 2) / 4]

Harga satuan kosong: 0


**Alasan** : Nilai harga_satuan yang kosong diisi menggunkana harga_dasar dari katalog produk berdasarkan produk_id. Cara ini digunakan untuk melengkapi harga yang hilang tanpa langsung menghapus transaksi. Pengisian ini mengasumsikan harga dasar dapat mewakili harga transaksi yang kosong

In [184]:
# penangan nilai negatif
df_trx_bersih = df_trx_bersih.filter(col("harga_satuan") > 0)
print("Baris setelah penanganan harga:", df_trx_bersih.count())

[Stage 481:============================>                            (2 + 2) / 4]

Baris setelah penanganan harga: 1991956


**Alasan** : Baris dengan harga nol atau negatif dihapus karena tidak memnunjukkan harga jual yang valid. Harga tersebut dapat menyebabkann hasil perhitungan nilai transaksi menjadi tidak tepat.

In [192]:
# menangani qty tidak wajar (outlier)
df_trx_bersih = df_trx_bersih.filter((col("qty") > 0) & (col("qty") <= batas_atas))

print("Baris setelah penangan outlier qty:", df_trx_bersih.count())

[Stage 502:============================>                            (2 + 2) / 4]

Baris setelah penangan outlier qty: 1990937


**Alasan** : Nilai qty yang mol, negatif, atau melebihi batas atas dihapus karena dianggap tidak wajar berdasarkan hasil analisis distribusi dankuantil pada TAhap 2. Hal ini dilakukan agar nilai transaksi tidak dipengaruhi oleh jumlah barang yang terlalu ekstrim.

In [190]:
# normaliasaikan nama kota
from pyspark.sql.functions import initcap, trim, col
df_trx_bersih = df_trx_bersih.withColumn("kota", initcap(trim(col("kota"))))
df_trx_bersih.select("kota").distinct().show(truncate=False)

[Stage 495:============================>                            (2 + 2) / 4]

+----------+
|kota      |
+----------+
|Semarang  |
|Solo      |
|Wonosobo  |
|Yogyakarta|
|Purworejo |
|Temanggung|
|Magelang  |
|Kebumen   |
+----------+



**Alasan** : Normalisasi dilakukan untuk menyamakan format penulisan nama kota. Fungsi trim() menghapus spasi berlebihan di awal dan akhir teks, sedangkan initcap() membuat huruf awal setiap kata menjanjadi kapital. Dengan demikian, validasi penulisan kota dapat dikurangi dan pengelomkan data menjadi lebih konsisten.

Menambah kolom turunan

In [200]:
from pyspark.sql.functions import month, to_date, dayofweek, when, col
df_trx_bersih = df_trx_bersih.withColumn("total_nilai", col("qty") * col("harga_satuan") * (1-col("diskon_persen") / 100))
df_trx_bersih = df_trx_bersih.withColumn("bulan", month(to_date(col("tanggal"))))
df_trx_bersih = df_trx_bersih.withColumn("hari_dalam_minggu", dayofweek(to_date(col("tanggal"))))
df_trx_bersih = df_trx_bersih.withColumn("flag_batal", when(col("status_pesanan") == "Dibatalkan", 1).otherwise(0))

df_trx_bersih.select("tanggal", "qty", "harga_satuan", "diskon_persen", "total_nilai", "bulan", "hari_dalam_minggu", "flag_batal").show(5, truncate=False)

[Stage 507:>                                                        (0 + 2) / 2]

+-------------------+---+------------+-------------+-----------+-----+-----------------+----------+
|tanggal            |qty|harga_satuan|diskon_persen|total_nilai|bulan|hari_dalam_minggu|flag_batal|
+-------------------+---+------------+-------------+-----------+-----+-----------------+----------+
|2026-06-25 11:25:45|7.0|750000.0    |0            |5250000.0  |6    |5                |0         |
|2026-05-04 13:10:39|1.0|60000.0     |15           |51000.0    |5    |2                |1         |
|2026-05-05 17:25:45|4.0|200000.0    |10           |720000.0   |5    |3                |1         |
|2026-05-08 00:57:07|4.0|85000.0     |5            |323000.0   |5    |6                |0         |
|2026-03-15 13:26:58|4.0|350000.0    |15           |1190000.0  |3    |1                |0         |
+-------------------+---+------------+-------------+-----------+-----+-----------------+----------+
only showing top 5 rows



Menyimpan data ke HDFS dalam format Parquet

In [204]:
!hdfs dfs -mkdir -p /tidarmard/curated

df_trx_bersih.write \
   .mode("overwrite") \
   .partitionBy("bulan") \
   .parquet("hdfs://localhost:9000/tidarmart/curated/transaksi")

print("Data transaksi bersih berhasil disimpan ke HDFS.")
!hdfs dfs -mkdir -p /tidarmard/curated/transaksi

[Stage 513:==========================================>              (3 + 1) / 4]

Data transaksi bersih berhasil disimpan ke HDFS.


In [205]:
df_trx_parquet = spark.read.parquet("hdfs://localhost:9000/tidarmart/curated/transaksi")

print("Jumlah baris hasil akhir:", df_trx_parquet.count())
df_trx_parquet.printSchema()
df_trx_parquet.show(5, truncate=False)

Jumlah baris hasil akhir: 1990937
root
 |-- produk_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- tanggal: string (nullable = true)
 |-- pelanggan_id: string (nullable = true)
 |-- qty: string (nullable = true)
 |-- harga_satuan: string (nullable = true)
 |-- diskon_persen: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- berat_kg: string (nullable = true)
 |-- jarak_km: string (nullable = true)
 |-- lama_kirim_hari: string (nullable = true)
 |-- status_pesanan: string (nullable = true)
 |-- total_nilai: double (nullable = true)
 |-- hari_dalam_minggu: integer (nullable = true)
 |-- flag_batal: integer (nullable = true)
 |-- bulan: integer (nullable = true)

+---------+----------+-------------------+------------+---+------------+-------------+-----------------+----------+--------+--------+---------------+--------------+-----------+-----------------+----------+-----+
|produk_id|order_id  |tangga

Membandingkan CSV dan Parquet

In [207]:
# ukuran penyimpanan 
!hdfs dfs -du -h /tidarmart/raw/transaksi
!hdfs dfs -du -h /tidarmart/curated/transaksi

46.8 M  46.8 M  /tidarmart/raw/transaksi/transaksi_part1.csv
46.8 M  46.8 M  /tidarmart/raw/transaksi/transaksi_part2.csv
46.8 M  46.8 M  /tidarmart/raw/transaksi/transaksi_part3.csv
46.8 M  46.8 M  /tidarmart/raw/transaksi/transaksi_part4.csv
0      0       /tidarmart/curated/transaksi/_SUCCESS
8.8 M  26.4 M  /tidarmart/curated/transaksi/bulan=1
8.0 M  24.1 M  /tidarmart/curated/transaksi/bulan=2
8.9 M  26.6 M  /tidarmart/curated/transaksi/bulan=3
8.6 M  25.7 M  /tidarmart/curated/transaksi/bulan=4
8.8 M  26.5 M  /tidarmart/curated/transaksi/bulan=5
8.6 M  25.7 M  /tidarmart/curated/transaksi/bulan=6


In [210]:
# Waktu  eksekusi query

import time

mulai = time.time()
df_trx.groupBy("kota").count().show()
waktu_csv = time.time() - mulai

mulai = time.time()
df_trx_parquet.groupBy("kota").count().show()
waktu_parquet = time.time() - mulai

print("Waktu CSV:", waktu_csv)
print("Waktu Parquet:", waktu_parquet)

+----------+------+
|      kota| count|
+----------+------+
|  Semarang|238446|
| PURWOREJO|  6311|
|YOGYAKARTA|  6340|
|      SOLO|  6335|
|      solo|  6367|
|  magelang|  6240|
|  wonosobo|  6277|
|yogyakarta|  6289|
| Purworejo|238337|
|   KEBUMEN|  6280|
|      Solo|238603|
|  Wonosobo|239362|
|Yogyakarta|238364|
|TEMANGGUNG|  6199|
|  semarang|  6374|
|  SEMARANG|  6254|
|Temanggung|238968|
|   Kebumen|238457|
|  Magelang|238449|
|  MAGELANG|  6379|
+----------+------+
only showing top 20 rows

+----------+------+
|      kota| count|
+----------+------+
|  Semarang|248658|
| Purworejo|248656|
|Yogyakarta|248576|
|  Wonosobo|249700|
|      Solo|249034|
|Temanggung|248983|
|   Kebumen|248620|
|  Magelang|248710|
+----------+------+

Waktu CSV: 0.9410049915313721
Waktu Parquet: 0.2932248115539551


In [212]:
# laporan kualitas data sesudah pembersihan
print("jumlah baris:", df_trx_bersih.count())
print("order_id unik:", df_trx_bersih.select("order_id").distinct().count())
print("qty kosong:", df_trx_bersih.filter("qty IS NULL").count())
print("Harga negatif:", df_trx_bersih.filter("harga_satuan < 0").count())
print("Qty tidak wajar:", df_trx_bersih.filter((col("qty") <= 0) | (col("qty") > batas_atas)).count())

jumlah baris: 1990937


order_id unik: 1990937


qty kosong: 0


Harga negatif: 0


[Stage 558:>                                                        (0 + 2) / 2]

Qty tidak wajar: 0


**Tahap 4 - Analitik Batch (DataFrame,  Join,  Window,  SparkSQL)**

pada tahap ini, dilakukan anlisis terhadap data transaksi yang sudah dibersihkan untuk mengetahui pola penjualan, omzet, produk terlais, dan perilaku pelanggan menggunakan PySpark.

In [215]:
from pyspark.sql import functions  as F
from pyspark.sql.window import Window

df_trx = spark.read.parquet("hdfs://localhost:9000/tidarmart/curated/transaksi")
df_trx = df_trx.withColumn("tanggal_ts", F.to_timestamp("tanggal"))
df_trx_selesai = df_trx.filter(F.col("status_pesanan") == "Selesai")

A. Gabungkan transaksi dengan produk dan pelanggan (join). Tampilkan omzet per kategori per kota dan jelaskan
strategi join yang dipakai (apakah perlu broadcast, mengapa).

In [219]:
# mengabungkan transakis produk dan pelanggan
df_join = (df_trx_selesai.join(F.broadcast(df_prod.select("produk_id", "nama_produk", "kategori")), "produk_id")
           .join(F.broadcast(df_pel.select("pelanggan_id", "membership")), "pelanggan_id"))

df_join.show(5, truncate=False)
print("Jumlah data setelah join:", df_join.count())

+------------+---------+----------+-------------------+---+------------+-------------+-----------------+----------+--------+--------+---------------+--------------+-----------+-----------------+----------+-----+-------------------+-----------------+------------+----------+
|pelanggan_id|produk_id|order_id  |tanggal            |qty|harga_satuan|diskon_persen|metode_pembayaran|kota      |berat_kg|jarak_km|lama_kirim_hari|status_pesanan|total_nilai|hari_dalam_minggu|flag_batal|bulan|tanggal_ts         |nama_produk      |kategori    |membership|
+------------+---------+----------+-------------------+---+------------+-------------+-----------------+----------+--------+--------+---------------+--------------+-----------+-----------------+----------+-----+-------------------+-----------------+------------+----------+
|29661       |2881     |ORD4000007|2026-03-10 07:37:59|6.0|15000.0     |5            |Transfer Bank    |Purworejo |35.64   |45.6    |4              |Selesai       |85500.0    |3 

In [220]:
# Tampilkan omzet per kategori per kota
df_analisis_a = (df_join.groupBy("kota", "kategori").agg(F.sum("total_nilai").alias("omzet")).orderBy("kota", F.desc("omzet")))
df_analisis_a.show(30, truncate=False)

[Stage 572:>                                                        (0 + 2) / 2]

+---------+------------+--------------+
|kota     |kategori    |omzet         |
+---------+------------+--------------+
|Kebumen  |Kesehatan   |2.74161165E10 |
|Kebumen  |Olahraga    |2.6913971E10  |
|Kebumen  |Fashion     |2.652292475E10|
|Kebumen  |Elektronik  |2.62478665E10 |
|Kebumen  |Makanan     |2.4419319E10  |
|Kebumen  |Rumah Tangga|2.3885337E10  |
|Kebumen  |Buku        |2.2828627E10  |
|Kebumen  |Otomotif    |2.2746731E10  |
|Magelang |Kesehatan   |2.926192975E10|
|Magelang |Fashion     |2.8231103E10  |
|Magelang |Olahraga    |2.724737325E10|
|Magelang |Elektronik  |2.699542525E10|
|Magelang |Rumah Tangga|2.5250143E10  |
|Magelang |Makanan     |2.463607125E10|
|Magelang |Buku        |2.45919395E10 |
|Magelang |Otomotif    |2.3750052E10  |
|Purworejo|Kesehatan   |2.8909868E10  |
|Purworejo|Fashion     |2.817392825E10|
|Purworejo|Elektronik  |2.715894625E10|
|Purworejo|Olahraga    |2.682064125E10|
|Purworejo|Rumah Tangga|2.5114211E10  |
|Purworejo|Makanan     |2.477725525E10|


Strategi join yang digunakan adalah Broadcast Join dengan fungsi F.broadcast(). Metode dipilih ini karena tabel produk dan pelangggan memiliki jumlah data lebih kecil dibandingkan tabel transaksi. Dengan broadcast,tabel berukuran kecil dikirimke setiap partisi pemrosesan sehingga dapat mengurangi shuffle saat proses penggabungan data.

B. Tren omzet bulanan dan pertumbuhan dibanding bulan sebelumnya (window function lag).

In [221]:
df_omzet_bulanan = ( df_trx_selesai.withColumn("periode_bulan", F.date_format("tanggal_ts", "yyyy-MM"))
                     .groupBy("periode_bulan").agg(F.sum("total_nilai").alias("total_omzet")).orderBy("periode_bulan"))

df_omzet_bulanan.show(20, truncate=False)

[Stage 575:>                                                        (0 + 2) / 2]

+-------------+---------------+
|periode_bulan|total_omzet    |
+-------------+---------------+
|2026-01      |2.84424408E11  |
|2026-02      |2.55630563E11  |
|2026-03      |2.85424765E11  |
|2026-04      |2.74086379E11  |
|2026-05      |2.8408596275E11|
|2026-06      |2.754526485E11 |
+-------------+---------------+



In [224]:
from pyspark.sql.window import Window

window_bulan = Window.orderBy("periode_bulan")

df_pertumbuhan = (df_omzet_bulanan.withColumn("omzet_sebelumnya", F.lag("total_omzet").over(window_bulan)).withColumn("pertumbuhan_persen",
                 F.round(((F.col("total_omzet") - F.col("omzet_sebelumnya")) / F.col("omzet_sebelumnya")) * 100, 2)))

df_pertumbuhan.show(20, truncate=False)

[Stage 578:>                                                        (0 + 2) / 2]

+-------------+---------------+----------------+------------------+
|periode_bulan|total_omzet    |omzet_sebelumnya|pertumbuhan_persen|
+-------------+---------------+----------------+------------------+
|2026-01      |2.84424408E11  |NULL            |NULL              |
|2026-02      |2.55630563E11  |2.84424408E11   |-10.12            |
|2026-03      |2.85424765E11  |2.55630563E11   |11.66             |
|2026-04      |2.74086379E11  |2.85424765E11   |-3.97             |
|2026-05      |2.8408596275E11|2.74086379E11   |3.65              |
|2026-06      |2.754526485E11 |2.8408596275E11 |-3.04             |
+-------------+---------------+----------------+------------------+



C. Peringkat 5 produk terlaris di setiap kota berdasarkan omzet (window function rank / dense_rank dengan
partisi).

In [227]:
window_produk = Window.partitionBy("kota").orderBy(F.desc("omzet"))

df_produk_kota = (df_join.groupBy("kota", "nama_produk").agg(F.sum("total_nilai").alias("omzet"))
    .withColumn("peringkat", F.dense_rank().over(window_produk)).filter(F.col("peringkat") <= 5).orderBy("kota", "peringkat"))


df_produk_kota.show(30, truncate=False)

[Stage 592:>                                                        (0 + 2) / 2]

+----------+-----------------+---------+---------+
|kota      |nama_produk      |omzet    |peringkat|
+----------+-----------------+---------+---------+
|Kebumen   |Rumah Tangga-2520|3.1395E8 |1        |
|Kebumen   |Otomotif-461     |3.0735E8 |2        |
|Kebumen   |Olahraga-775     |2.9955E8 |3        |
|Kebumen   |Elektronik-2982  |2.91E8   |4        |
|Kebumen   |Olahraga-1162    |2.85E8   |5        |
|Magelang  |Rumah Tangga-2531|3.6945E8 |1        |
|Magelang  |Kesehatan-4284   |3.5535E8 |2        |
|Magelang  |Rumah Tangga-192 |3.20925E8|3        |
|Magelang  |Fashion-4168     |3.1755E8 |4        |
|Magelang  |Fashion-1180     |3.11175E8|5        |
|Purworejo |Rumah Tangga-192 |3.25725E8|1        |
|Purworejo |Fashion-3573     |3.2445E8 |2        |
|Purworejo |Makanan-521      |3.14025E8|3        |
|Purworejo |Fashion-1125     |3.04125E8|4        |
|Purworejo |Kesehatan-1528   |3.0255E8 |5        |
|Semarang  |Olahraga-2272    |3.41625E8|1        |
|Semarang  |Fashion-4673     |3

D. Omzet kumulatif (running total) per kategori sepanjang waktu serta rata-rata bergerak 7 hari (window dengan
rentang baris).

In [237]:
#menghitung omzet harian per kategori
df_omzet_harian = (df_join.groupBy("kategori", "tanggal_ts").agg(F.sum("total_nilai").alias("omzet_harian")).orderBy("kategori", "tanggal_ts"))

df_omzet_harian.show(10, truncate=False)

[Stage 613:>                                                        (0 + 2) / 2]

+--------+-------------------+------------+
|kategori|tanggal_ts         |omzet_harian|
+--------+-------------------+------------+
|Buku    |2026-01-01 00:01:51|114000.0    |
|Buku    |2026-01-01 00:01:56|272000.0    |
|Buku    |2026-01-01 00:07:36|297500.0    |
|Buku    |2026-01-01 00:07:50|25000.0     |
|Buku    |2026-01-01 00:08:17|408000.0    |
|Buku    |2026-01-01 00:08:38|600000.0    |
|Buku    |2026-01-01 00:08:42|336000.0    |
|Buku    |2026-01-01 00:10:23|280000.0    |
|Buku    |2026-01-01 00:10:40|1800000.0   |
|Buku    |2026-01-01 00:11:31|180000.0    |
+--------+-------------------+------------+
only showing top 10 rows



In [238]:
df_omzet_harian.select("kategori").distinct().show()

[Stage 618:>                                                        (0 + 2) / 2]

+------------+
|    kategori|
+------------+
|    Olahraga|
|     Makanan|
|  Elektronik|
|    Otomotif|
|Rumah Tangga|
|        Buku|
|     Fashion|
|   Kesehatan|
+------------+



In [239]:
# menghitung omzet kumulatif dan rata-rata  bergerak 7 hari

window_kumulatif = (Window.partitionBy("kategori").orderBy("tanggal_ts").rowsBetween(Window.unboundedPreceding, 0))

window_7hari = (Window.partitionBy("kategori").orderBy("tanggal_ts").rowsBetween(-6,0))

df_analisis_d = (df_omzet_harian.withColumn("omzet_kumulatif", F.sum("omzet_harian").over(window_kumulatif)).withColumn("rata_rata_7hari",
                F.avg("omzet_harian").over(window_7hari)))

df_analisis_d.show(20, truncate=False)

[Stage 633:===================>                                     (1 + 2) / 3]

+----------+-------------------+------------+---------------+------------------+
|kategori  |tanggal_ts         |omzet_harian|omzet_kumulatif|rata_rata_7hari   |
+----------+-------------------+------------+---------------+------------------+
|Elektronik|2026-01-01 00:01:03|1750000.0   |1750000.0      |1750000.0         |
|Elektronik|2026-01-01 00:03:32|840000.0    |2590000.0      |1295000.0         |
|Elektronik|2026-01-01 00:07:01|90000.0     |2680000.0      |893333.3333333334 |
|Elektronik|2026-01-01 00:09:28|1500000.0   |4180000.0      |1045000.0         |
|Elektronik|2026-01-01 00:10:10|1400000.0   |5580000.0      |1116000.0         |
|Elektronik|2026-01-01 00:11:23|180000.0    |5760000.0      |960000.0          |
|Elektronik|2026-01-01 00:12:43|750000.0    |6510000.0      |930000.0          |
|Elektronik|2026-01-01 00:13:16|60000.0     |6570000.0      |688571.4285714285 |
|Elektronik|2026-01-01 00:14:11|90000.0     |6660000.0      |581428.5714285715 |
|Elektronik|2026-01-01 00:14

E. Satu analisis yang sama dengan salah satu analisis A–D, ditulis ulang menggunakan Spark SQL
(createOrReplaceTempView) dan buktikan hasilnya identik. Bandingkan rencana eksekusi dengan explain().

Pada tahap ini, analisis tren omzet bulanann yang sebelumnya dilakukan menggunakan DataFrame akan ditulis ulang menggunakan Spark SQL. HAsil kedua metode kemudian dibandingkan untul memastikan hasil analisisnya sama.

In [244]:
# menulis ulang omzet bulanan(analisis b) menggunakan Spark SQL

df_trx_selesai.createOrReplaceTempView("transaksi_selesai")

df_omzet_sql = spark.sql("""SELECT date_format(tanggal_ts, 'yyyy-MM') AS periode_bulan, 
                          SUM(total_nilai) AS total_omzet 
                          FROM transaksi_selesai
                          GROUP BY date_format(tanggal_ts, 'yyyy-MM') 
                          ORDER BY periode_bulan""")

df_omzet_sql.show(20, truncate=False)                            

[Stage 638:>                                                        (0 + 2) / 2]

+-------------+---------------+
|periode_bulan|total_omzet    |
+-------------+---------------+
|2026-01      |2.84424408E11  |
|2026-02      |2.55630563E11  |
|2026-03      |2.85424765E11  |
|2026-04      |2.74086379E11  |
|2026-05      |2.8408596275E11|
|2026-06      |2.754526485E11 |
+-------------+---------------+



In [246]:
# buktikan apakah sama dengan analisisn poin b
print("Perbedaan DataFrame dengan SQL:")
df_omzet_bulanan.exceptAll(df_omzet_sql).show()
print("Perbedaan DataFrame dengan SQL:")
df_omzet_sql.exceptAll(df_omzet_bulanan).show()

Perbedaan DataFrame dengan SQL:
+-------------+-----------+
|periode_bulan|total_omzet|
+-------------+-----------+
+-------------+-----------+

Perbedaan DataFrame dengan SQL:
+-------------+-----------+
|periode_bulan|total_omzet|
+-------------+-----------+
+-------------+-----------+



In [247]:
print("Execution Plan DataFrame:")
df_omzet_bulanan.explain()
print("Execution Plain Spark SQL:")
df_omzet_sql.explain()

Execution Plan DataFrame:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [periode_bulan#4388 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(periode_bulan#4388 ASC NULLS FIRST, 4), ENSURE_REQUIREMENTS, [plan_id=13345]
      +- HashAggregate(keys=[periode_bulan#4388], functions=[sum(total_nilai#4067)])
         +- Exchange hashpartitioning(periode_bulan#4388, 4), ENSURE_REQUIREMENTS, [plan_id=13342]
            +- HashAggregate(keys=[periode_bulan#4388], functions=[partial_sum(total_nilai#4067)])
               +- Project [total_nilai#4067, date_format(cast(tanggal#4056 as timestamp), yyyy-MM, Some(Asia/Jakarta)) AS periode_bulan#4388]
                  +- Filter (isnotnull(status_pesanan#4066) AND (status_pesanan#4066 = Selesai))
                     +- FileScan parquet [tanggal#4056,status_pesanan#4066,total_nilai#4067,bulan#4070] Batched: true, DataFilters: [isnotnull(status_pesanan#4066), (status_pesanan#4066 = Selesai)], Format: Parquet, Location: InMemo

F. Apakah diskon yang lebih besar meningkatkan qty rata-rata dan omzet per pesanan?
Analisis per kategori, termasuk kategori yang tidak responsif terhadap diskon.

In [249]:
# menghitung rentang diskon
df_join.agg(F.min("diskon_persen").alias("diskon_minimum"),
            F.max("diskon_persen").alias("diskon_maksimum"),
            F.countDistinct("diskon_persen").alias("jumlah_variasi_diskon")).show() 

[Stage 679:>                                                        (0 + 2) / 2]

+--------------+---------------+---------------------+
|diskon_minimum|diskon_maksimum|jumlah_variasi_diskon|
+--------------+---------------+---------------------+
|             0|              5|                    5|
+--------------+---------------+---------------------+



In [254]:
df_join = df_join.withColumn("diskon_persen", F.col("diskon_persen").cast("double")) #ubah tipe datanya dari string ke double

df_join.select("diskon_persen") \
       .distinct() \
       .orderBy("diskon_persen") \
       .show()

[Stage 702:>                                                        (0 + 2) / 2]

+-------------+
|diskon_persen|
+-------------+
|          0.0|
|          5.0|
|         10.0|
|         15.0|
|         20.0|
+-------------+



In [ ]:
# periksa tipe data qty dan total nilai agar perhitungantidak error

df_join.select("qty", "total_nilai").print

G. Temukan produk “bintang tersembunyi”: produk dengan tingkat pengembalian rendah
tetapi omzet tinggi, dan produk bermasalah (pengembalian tinggi).

H. Analisis pola hari dalam seminggu dan bulan untuk tiap kategori. Kapan waktu terbaik
untuk promosi masing-masing kategori?